# 第15章  随机变量与分布 —— 用采样代替公式

> 离散(PMF)+连续(PDF)+均匀+伯努利四种分布。MLE直觉。四种采样策略(Greedy/Temperature/Top-k/Top-p)——LLM文本生成最后一公里。
> 第1章(NumPy/matplotlib)、第3章(标量与向量)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('环境就绪')


---
## 15.1-15.4  四种核心分布

离散PMF:骰子频率收敛。连续PDF:正态68-95-99.7。均匀:Xavier初始化。伯努利:二分类/Dropout。

AI连接：Softmax输出=50000维PMF(Ch29)。正态=AI默认噪声模型(Ch21/Ch25)。


In [ ]:
import numpy as np
from math import erf
np.random.seed(42)
# Dice frequency convergence
for n in[30,300,3000]:
  r=np.random.randint(1,7,size=n)
  f=np.array([np.sum(r==k)/n for k in range(1,7)])
  print(f'n={n:4d}: max_err={np.max(np.abs(f-1/6)):.4f}')
# Normal 68-95-99.7
def ncdf(x): return 0.5*(1+erf(x/1.4142135623730951))
s=np.random.randn(10000)
for k in[1,2,3]:
  print(f'+/-{k}sig: actual={np.mean(np.abs(s)<k):.3f} theory={ncdf(k)-ncdf(-k):.3f}')
# Xavier uniform init
fan_in,fan_out=784,256; limit=np.sqrt(6/(fan_in+fan_out))
w=np.random.uniform(-limit,limit,(fan_in,fan_out))
print(f'Xavier: mean={w.mean():.6f} var={w.var():.6f} (theory={(2*limit)**2/12:.6f})')


---
## 15.5  极大似然估计(MLE)

选择使数据出现概率最大的参数。正态:mu=样本均值。交叉熵损失=在做MLE(第19章)。


In [ ]:
import numpy as np
np.random.seed(42)
data=np.random.randn(200)*2.0+5.0
cm=np.cumsum(data)/np.arange(1,201)
print(f'n=5: mu_hat={data[:5].mean():.2f}, n=200: mu_hat={data.mean():.3f} (true=5.0)')
print('More data -> MLE more accurate')


---
## 15.6  采样策略 🆕

Greedy=argmax(重复)。Temperature=softmax(x/T):T->0贪心,T->无穷均匀。Top-k=前k候选。Top-p=累积>=p的动态候选集,工业标准0.9。

Agent视角框:工具调用T=0.1~0.3,创意规划T=0.7~0.9,反思高温度多轨迹。
AI连接：第33章GPT-2四种策略生成对比。


In [ ]:
import numpy as np
logits=np.array([3.,2.,1.5,1.,0.5,0.2,0.1,0.05,0.02,0.01])
def sm(x,T=1.):
  x=np.array(x,dtype=np.float64)/T
  return np.exp(x-x.max())/np.exp(x-x.max()).sum()
orig=sm(logits)
print(f'T=0.5: peak={sm(logits,0.5)[0]:.3f} vs orig={orig[0]:.3f} (more peaked)')
print(f'T=2.0: peak={sm(logits,2.0)[0]:.3f} vs orig={orig[0]:.3f} (flatter)')
print(f'T->0:  peak={sm(logits,0.01)[0]:.3f} (->greedy)')
# Top-k=3
tk=orig.copy(); tk[np.argsort(tk)[:-3]]=0; tk/=tk.sum()
print(f'Top-k=3: nonzero={np.count_nonzero(tk)}')
# Top-p=0.9
si=np.argsort(orig)[::-1]; cut=np.searchsorted(np.cumsum(orig[si]),0.9)+1
tp=orig.copy(); tp[si[cut:]]=0; tp/=tp.sum()
print(f'Top-p=0.9: kept {np.count_nonzero(tp)} tokens')


---
## 习题

> 在下方新建代码单元格作答。

1.(概念)离散vs连续随机变量区别？
2.(概念)68-95-99.7规则？异常检测怎么用？
3.(概念)MLE核心直觉,一句话。
4.(代码)两骰子之和分布(10000次),理论vs模拟。
5.(代码)T=0.1 vs T=0.9同prompt各5条回复,对比多样性和重复率。
6.(代码)五种采样策略各2000次,2x3子图对比频率。


---

> 章末钩子：事件不是孤立的——已知一件事发生,另一件事概率怎么变？
预览：下一章——**条件概率与贝叶斯定理**。

> 提示：完成后运行 Kernel -> Restart & Run All 验证。
